# Caso 6 — Análise por Consulta

Selecionamos, a partir das diferenças de AP calculadas no Caso 5:

- **2 consultas em que o BM25 é claramente superior** ao Modelo Vetorial;
- **2 consultas em que o Modelo Vetorial é claramente superior** ao BM25;
- **2 consultas em que ambos os modelos têm desempenho insatisfatório**.

Para cada uma, mostramos os 5 primeiros documentos retornados por cada
modelo, indicando quais são de fato relevantes segundo o qrels.


In [1]:
from pathlib import Path
import sys
project_root = Path.cwd()
if not (project_root / "src").is_dir() and (project_root.parent / "src").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import pickle
import pandas as pd
from IPython.display import display

from src.cranfield_data import load_cranfield

df_docs, df_queries, df_qrels = load_cranfield()
doc_title = dict(zip(df_docs.doc_id, df_docs.title.str.replace("\n", " ")))
query_text = dict(zip(df_queries.query_id, df_queries.text.str.replace("\n", " ")))

RESULTS_DIR = project_root / "data" / "processed"
vsm_eval = pd.read_csv(RESULTS_DIR / "vsm_perquery.csv", dtype={"query_id": str}).set_index("query_id")
bm25_eval = pd.read_csv(RESULTS_DIR / "bm25_perquery.csv", dtype={"query_id": str}).set_index("query_id")
with open(RESULTS_DIR / "vsm_rankings.pkl", "rb") as f:
    vsm_rankings = pickle.load(f)
with open(RESULTS_DIR / "bm25_rankings.pkl", "rb") as f:
    bm25_rankings = pickle.load(f)


def show_top_n(qid, n=5):
    """Mostra os top-n documentos de cada modelo para uma consulta, com o
    grau de relevância (qrels) de cada documento retornado."""
    grades = dict(zip(df_qrels[df_qrels.query_id == qid].doc_id,
                       df_qrels[df_qrels.query_id == qid].relevance))
    print(f"Consulta {qid}: {query_text[qid]}")
    print(f"AP -> BM25: {bm25_eval.loc[qid, 'AP']:.3f} | "
          f"Modelo Vetorial: {vsm_eval.loc[qid, 'AP']:.3f}")
    frames = {}
    for nome, rankings in [("BM25", bm25_rankings), ("Modelo Vetorial", vsm_rankings)]:
        rows = []
        for rank, did in enumerate(rankings[qid][:n], start=1):
            grade = grades.get(did)
            rows.append({
                "rank": rank,
                "doc_id": did,
                "grau": grade if grade is not None else "não julgado",
                "relevante?": "SIM" if (grade is not None and grade >= 1) else "não",
                "titulo": doc_title[did][:75],
            })
        frames[nome] = pd.DataFrame(rows).set_index("rank")
    for nome, frame in frames.items():
        print(f"-- Top-{n} {nome} --")
        display(frame)
    print()


[cranfield_data] ir_datasets indisponível (RuntimeError: ('All download sources failed', [(RequestsDownload('http://ir.dcs.gla.ac.uk/resources/test_collections/cran/cran.tar.gz', tries=None), HTTPError('403 Client Error: Forbidden for url: http://ir.dcs.gla.ac.uk/resources/test_collections/cran/cran.tar.gz')), (RequestsDownload('https://mirror.ir-datasets.com/1730f7be572d95a5a4b56c59a7b900a5', tries=None), HTTPError('403 Client Error: Forbidden for url: https://mirror.ir-datasets.com/1730f7be572d95a5a4b56c59a7b900a5'))])). Usando arquivos locais em data/raw/ como fallback.


[WARNING] Download failed: 403 Client Error: Forbidden for url: http://ir.dcs.gla.ac.uk/resources/test_collections/cran/cran.tar.gz
[WARNING] Download failed: 403 Client Error: Forbidden for url: https://mirror.ir-datasets.com/1730f7be572d95a5a4b56c59a7b900a5


## 1) Consultas em que o BM25 é claramente superior

**Consulta 167** (AP: BM25 0,750 vs. Vetorial 0,225): o Modelo Vetorial
coloca em 1º lugar um documento não julgado (553) que repete o termo
`ablat` 7 vezes em um texto curto, empurrando para a 4ª posição o
documento realmente relevante (274). O BM25, graças à saturação de
frequência de termo, não deixa essa repetição dominar o score e classifica
274 em 1º lugar (ver mecanismo detalhado no Caso 5).

**Consulta 173** (AP: BM25 1,000 vs. Vetorial 0,583): os dois documentos
relevantes (367 e 451, sobre o "método de Lyapunov") existem, mas o
Modelo Vetorial insere na 1ª posição o documento 532 — não relevante
(grau -1) — que também menciona "estabilidade" e "segundo método", por
coincidência lexical. O BM25 ainda recupera 532 na 3ª posição (não é
imune ao mesmo problema), mas pondera menos essa coincidência e consegue
colocar os dois documentos relevantes exatamente nas posições 1 e 2,
obtendo AP perfeito.


In [2]:
for qid in ["167", "173"]:
    show_top_n(qid)


Consulta 167: exact solution methods for calculating the ablative mass loss of a material ablating at high temperatures in a hypersonic flight environment .
AP -> BM25: 0.750 | Modelo Vetorial: 0.225
-- Top-5 BM25 --


,doc_id,grau,relevante?,titulo
rank,,,,
1,274,2,SIM,analysis of quartz and teflon shields for a pa...
2,1279,não julgado,não,sublimation in a hypersonic environment .
3,553,não julgado,não,ablation of glassy materials around blunt bodi...
4,82,3,SIM,theoretical investigation of the ablation of a...
5,1098,não julgado,não,an experimental investigation of ablating mate...


-- Top-5 Modelo Vetorial --


,doc_id,grau,relevante?,titulo
rank,,,,
1,553,não julgado,não,ablation of glassy materials around blunt bodi...
2,1279,não julgado,não,sublimation in a hypersonic environment .
3,1099,não julgado,não,a theoretical study of stagnation point ablati...
4,274,2,SIM,analysis of quartz and teflon shields for a pa...
5,1100,não julgado,não,an analytical investigation of ablation .



Consulta 173: references on lyapunov's method on the stability of linear differential equations with periodic coefficients .
AP -> BM25: 1.000 | Modelo Vetorial: 0.583
-- Top-5 BM25 --


,doc_id,grau,relevante?,titulo
rank,,,,
1,367,1,SIM,control system and analysis and design via the...
2,451,1,SIM,liapunov's methods in automatic control theory .
3,532,-1,não,pitch-yaw stability of a missile oscillating i...
4,917,não julgado,não,a method of calculating the short period longi...
5,767,não julgado,não,mathematical techniques applying to the therma...


-- Top-5 Modelo Vetorial --


,doc_id,grau,relevante?,titulo
rank,,,,
1,532,-1,não,pitch-yaw stability of a missile oscillating i...
2,367,1,SIM,control system and analysis and design via the...
3,451,1,SIM,liapunov's methods in automatic control theory .
4,251,não julgado,não,a collection of longitudinal stability derivat...
5,1067,não julgado,não,plastic stability theory of geometrically orth...


## 2) Consultas em que o Modelo Vetorial é claramente superior

**Consulta 113** (AP: Vetorial 0,514 vs. BM25 0,149): O BM25 acumula
score para o documento 815 (não julgado) somando contribuições de vários
termos **genéricos** da consulta (`mach`, `number`, `aerodynam`) que
aparecem com frequência razoável, mas têm idf baixo por serem comuns na
coleção. O Modelo Vetorial encontra rapidamente os dois documentos
relevantes (265 e 748), pois a similaridade de cosseno favorece o
documento 265 — curto e concentrado em termos mais raros e específicos da
consulta (`oscillatori`, `control`, `surfac`) — ver Caso 5 para os números
de sobreposição.

**Consulta 148** (AP: Vetorial 0,450 vs. BM25 0,210): padrão semelhante —
os documentos relevantes 1048 e 1050 aparecem nas 2 primeiras posições do
Modelo Vetorial, mas só nas posições 3 e 4 do BM25, que insere antes o
documento 956 (não relevante, grau -1) sobre um tema visualmente muito
próximo ("cilindros sanduíche corrugados").


In [3]:
for qid in ["113", "148"]:
    show_top_n(qid)


Consulta 113: what data exists on oscillatory aerodynamic forces on control surfaces at transonic mach numbers .
AP -> BM25: 0.149 | Modelo Vetorial: 0.514
-- Top-5 BM25 --


,doc_id,grau,relevante?,titulo
rank,,,,
1,704,não julgado,não,a systematic kernel function procedure for det...
2,815,não julgado,não,investigation of several blunt bodies to deter...
3,748,3,SIM,subsonic aerodynamic flutter derivatives for w...
4,638,não julgado,não,longitudinal aerodynamic characteristics at lo...
5,708,não julgado,não,aerodynamic characteristics of two winged reen...


-- Top-5 Modelo Vetorial --


,doc_id,grau,relevante?,titulo
rank,,,,
1,265,3,SIM,some instabilities arising from the interactio...
2,748,3,SIM,subsonic aerodynamic flutter derivatives for w...
3,594,não julgado,não,wind tunnel techniques for the measurements of...
4,1272,não julgado,não,oscillatory aerodynamic coefficients for a uni...
5,716,não julgado,não,study of the oscillatory motion of manned vehi...



Consulta 148: papers on small deflection theory for buckling of sandwich cylinders .
AP -> BM25: 0.210 | Modelo Vetorial: 0.450
-- Top-5 BM25 --


,doc_id,grau,relevante?,titulo
rank,,,,
1,956,-1,não,elastic stability of simply supported corrugat...
2,1048,2,SIM,a small deflection theory for curved sandwich ...
3,1126,não julgado,não,an engineer's conceptual approach to the buckl...
4,1050,2,SIM,compressive buckling of simply supported curve...
5,1127,não julgado,não,the buckling of sandwich type panels .


-- Top-5 Modelo Vetorial --


,doc_id,grau,relevante?,titulo
rank,,,,
1,1048,2,SIM,a small deflection theory for curved sandwich ...
2,1050,2,SIM,compressive buckling of simply supported curve...
3,956,-1,não,elastic stability of simply supported corrugat...
4,1126,não julgado,não,an engineer's conceptual approach to the buckl...
5,761,não julgado,não,buckling of sandwich under normal pressure .


## 3) Consultas em que ambos os modelos falham

**Consulta 31** ("que tamanho de placa de extremidade pode ser usado com
segurança para simular condições de escoamento bidimensional..."): possui
um único documento relevante (776, grau 4), que **nenhum dos dois modelos
coloca nem perto do Top-10** — ele aparece na posição 1080 (BM25) e 1001
(Modelo Vetorial) de 1400! Pior ainda: o documento 751 (explicitamente
**não relevante**, grau -1) — que tem forte sobreposição lexical com a
consulta ("end plates", "three dimensional flow") mas trata do problema
oposto (como *evitar* o efeito tridimensional, não como simulá-lo) — é
colocado em 1º lugar por **ambos** os modelos.

**Consulta 22** ("alguém mais descobriu que o atrito de pele turbulento
não é muito sensível à variação da viscosidade com a temperatura..."): o
único documento relevante (68, grau 1) fica na posição 607 em ambos os
rankings. O título do documento relevante ("some aspects of air-helium
simulation and hypersonic approximations") **não compartilha praticamente
nenhum termo de conteúdo** com a consulta — a relevância aparentemente
depende de uma ligação conceitual (mesma técnica experimental discutida
no corpo do texto) que não aparece no título nem é capturada por nenhum
modelo puramente lexical.

Em ambos os casos o problema não está em qual modelo foi escolhido — é o
clássico **problema de vocabulário** (sinonímia/paráfrase): quando a
consulta e o documento relevante descrevem o mesmo conceito com palavras
de superfície muito diferentes, nenhum modelo baseado em correspondência
exata de termos (bag-of-words) consegue recuperá-lo bem. Isso é retomado
com mais profundidade no Caso 9 (Análise de Erros).


In [4]:
for qid in ["31", "22"]:
    show_top_n(qid)


Consulta 31: what size of end plate can be safely used to simulate two-dimensional flow conditions over a bluff cylindrical body of finite aspect ratio .
AP -> BM25: 0.001 | Modelo Vetorial: 0.001
-- Top-5 BM25 --


,doc_id,grau,relevante?,titulo
rank,,,,
1,751,-1,não,a note on the use of end plates to prevent thr...
2,1209,não julgado,não,aerodynamic processes in the downwash-impingem...
3,1153,não julgado,não,a study of the simulation of flow with free st...
4,1245,não julgado,não,some aspects of nonequilibrium flows .
5,228,não julgado,não,navier-stokes solutions at large distances fro...


-- Top-5 Modelo Vetorial --


,doc_id,grau,relevante?,titulo
rank,,,,
1,751,-1,não,a note on the use of end plates to prevent thr...
2,1209,não julgado,não,aerodynamic processes in the downwash-impingem...
3,68,não julgado,não,some aspects of air-helium simulation and hype...
4,1153,não julgado,não,a study of the simulation of flow with free st...
5,176,não julgado,não,base pressure at subsonic speeds in the presen...



Consulta 22: did anyone else discover that the turbulent skin friction is not over sensitive to the nature of the variation of the viscosity with temperature .
AP -> BM25: 0.002 | Modelo Vetorial: 0.002
-- Top-5 BM25 --


,doc_id,grau,relevante?,titulo
rank,,,,
1,125,não julgado,não,measurements of skin friction of the compressi...
2,560,não julgado,não,a theoretical study of the effect of upstream ...
3,50,não julgado,não,investigation of laminar boundary layer in com...
4,413,não julgado,não,turbulent skin friction at high mach numbers a...
5,81,não julgado,não,compressible laminar flow and heat transfer ab...


-- Top-5 Modelo Vetorial --


,doc_id,grau,relevante?,titulo
rank,,,,
1,153,não julgado,não,"on the steady motion of viscous, incompressibl..."
2,207,não julgado,não,laminar boundary layer oscillations and transi...
3,125,não julgado,não,measurements of skin friction of the compressi...
4,413,não julgado,não,turbulent skin friction at high mach numbers a...
5,254,não julgado,não,boundary layers with suction and injection . a...
